# ⚡ پردازش موازی در Climatology Engine

این نوت‌بوک روش‌های پردازش موازی برای افزایش سرعت محاسبات را معرفی می‌کند.

**مواردی که یاد می‌گیرید:**
- مفهوم پردازش موازی و مزایای آن
- روش‌های مختلف پردازش موازی در Python
- استفاده از `multiprocessing` برای پردازش موازی
- استفاده از `joblib` برای پردازش موازی
- استفاده از `Dask` برای پردازش موازی
- مقایسه زمان اجرا در حالت‌های مختلف
- انتخاب بهترین روش برای داده‌های بزرگ

---

## 📐 نظریه پردازش موازی

پردازش موازی (Parallel Processing) روشی برای افزایش سرعت اجرای برنامه‌ها با استفاده همزمان از چندین هسته پردازنده است.

### مزایای پردازش موازی

| مزیت | توضیح |
|------|-------|
| **سرعت بیشتر** | کاهش زمان اجرا با تقسیم کار بین چندین پردازنده |
| **مقیاس‌پذیری** | امکان پردازش داده‌های بزرگ‌تر با افزایش هسته‌ها |
| **استفاده بهینه از منابع** | استفاده حداکثری از توان CPU |
| **پاسخ‌دهی بهتر** | اجرای همزمان چندین وظیفه |

### روش‌های پردازش موازی در Python

| روش | کاربرد | مزایا | معایب |
|-----|--------|-------|-------|
| `multiprocessing` | CPU-bound tasks | کنترل کامل بر فرآیندها | مدیریت دستی |
| `joblib` | داده‌های بزرگ | ساده و کارآمد | وابستگی به کتابخانه |
| `Dask` | داده‌های بسیار بزرگ | مقیاس‌پذیری بالا | پیچیدگی بیشتر |
| `Ray` | سیستم‌های توزیع‌شده | انعطاف‌پذیری بالا | نصب جداگانه |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بارگذاری پلاگین‌های توزیع
plugins = load_plugins()
print(f'✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# انتخاب داده tmean برای یک سال
data_year = data[:365, 1]

print(f'📊 تعداد داده‌ها: {len(data_year)}')
print(f'   میانگین: {np.mean(data_year):.2f}°C')
print(f'   انحراف معیار: {np.std(data_year):.2f}°C')

In [ ]:
# تعریف تابع برای پردازش موازی
def process_station(station_data, dist_name='Normal'):
    """
    پردازش یک ایستگاه و برازش توزیع
    """
    dist = distributions[dist_name]
    try:
        res = dist.fit(station_data)
        return {
            'status': 'success',
            'aicc': res.get('aicc', np.nan),
            'bic': res.get('bic', np.nan),
            'loglik': res.get('loglik', np.nan)
        }
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e)
        }

# تست تابع روی یک داده نمونه
test_result = process_station(data_year, 'Normal')
print("✅ تابع پردازش تعریف شد.")
print(f"   نتیجه تست: {test_result}")

In [ ]:
# ============================================================================
# ۱. پردازش سریال (بدون موازی‌سازی)
# ============================================================================

def serial_processing(data_list, dist_name='Normal'):
    """پردازش سریال داده‌ها"""
    results = []
    for data in data_list:
        res = process_station(data, dist_name)
        results.append(res)
    return results

# ایجاد داده‌های تست (۱۰۰ تکرار از داده نمونه)
test_data = [data_year + np.random.normal(0, 0.1, len(data_year)) for _ in range(100)]

print(f"📊 تعداد داده‌های تست: {len(test_data)}")
print(f"   اندازه هر داده: {len(test_data[0])}")

In [ ]:
# اجرای پردازش سریال و اندازه‌گیری زمان
print("\n🔄 در حال اجرای پردازش سریال...")
start_time = time.time()
serial_results = serial_processing(test_data, 'Normal')
serial_time = time.time() - start_time

success_count = sum(1 for r in serial_results if r.get('status') == 'success')
print(f"✅ پردازش سریال کامل شد.")
print(f"   زمان اجرا: {serial_time:.2f} ثانیه")
print(f"   تعداد موفق: {success_count} از {len(test_data)}")

In [ ]:
# ============================================================================
# ۲. پردازش موازی با multiprocessing
# ============================================================================

from multiprocessing import Pool, cpu_count

def parallel_processing_multiprocessing(data_list, dist_name='Normal', n_workers=None):
    """پردازش موازی با multiprocessing"""
    if n_workers is None:
        n_workers = cpu_count()
    
    with Pool(processes=n_workers) as pool:
        # ارسال داده‌ها به تابع پردازش
        results = pool.starmap(process_station, [(data, dist_name) for data in data_list])
    return results

print(f"✅ تعداد هسته‌های موجود: {cpu_count()}")

# اجرای پردازش موازی با multiprocessing
print("\n🔄 در حال اجرای پردازش موازی با multiprocessing...")
start_time = time.time()
mp_results = parallel_processing_multiprocessing(test_data, 'Normal')
mp_time = time.time() - start_time

success_count = sum(1 for r in mp_results if r.get('status') == 'success')
print(f"✅ پردازش موازی با multiprocessing کامل شد.")
print(f"   زمان اجرا: {mp_time:.2f} ثانیه")
print(f"   تعداد موفق: {success_count} از {len(test_data)}")

In [ ]:
# ============================================================================
# ۳. پردازش موازی با joblib
# ============================================================================

try:
    from joblib import Parallel, delayed
    
    def parallel_processing_joblib(data_list, dist_name='Normal', n_workers=-1):
        """پردازش موازی با joblib"""
        if n_workers == -1:
            n_workers = cpu_count()
        
        results = Parallel(n_jobs=n_workers, verbose=0)(
            delayed(process_station)(data, dist_name) for data in data_list
        )
        return results

    print("\n🔄 در حال اجرای پردازش موازی با joblib...")
    start_time = time.time()
    joblib_results = parallel_processing_joblib(test_data, 'Normal')
    joblib_time = time.time() - start_time

    success_count = sum(1 for r in joblib_results if r.get('status') == 'success')
    print(f"✅ پردازش موازی با joblib کامل شد.")
    print(f"   زمان اجرا: {joblib_time:.2f} ثانیه")
    print(f"   تعداد موفق: {success_count} از {len(test_data)}")
    
    joblib_available = True
except ImportError:
    print("⚠️ کتابخانه joblib نصب نیست. برای نصب: pip install joblib")
    joblib_time = None
    joblib_available = False

In [ ]:
# ============================================================================
# ۴. پردازش موازی با Dask
# ============================================================================

try:
    import dask
    from dask import delayed, compute
    import dask.multiprocessing
    
    def parallel_processing_dask(data_list, dist_name='Normal'):
        """پردازش موازی با Dask"""
        lazy_results = [delayed(process_station)(data, dist_name) for data in data_list]
        results = compute(*lazy_results, scheduler='multiprocessing')
        return list(results)

    print("\n🔄 در حال اجرای پردازش موازی با Dask...")
    start_time = time.time()
    dask_results = parallel_processing_dask(test_data, 'Normal')
    dask_time = time.time() - start_time

    success_count = sum(1 for r in dask_results if r.get('status') == 'success')
    print(f"✅ پردازش موازی با Dask کامل شد.")
    print(f"   زمان اجرا: {dask_time:.2f} ثانیه")
    print(f"   تعداد موفق: {success_count} از {len(test_data)}")
    
    dask_available = True
except ImportError:
    print("⚠️ کتابخانه Dask نصب نیست. برای نصب: pip install dask")
    dask_time = None
    dask_available = False

In [ ]:
# ============================================================================
# ۵. مقایسه زمان اجرا
# ============================================================================

comparison_data = {
    'روش': ['سریال', 'multiprocessing', 'joblib', 'Dask'],
    'زمان (ثانیه)': [serial_time, mp_time, joblib_time if joblib_available else np.nan, 
                    dask_time if dask_available else np.nan]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.dropna()

print("📊 مقایسه زمان اجرا:")
print("=" * 60)
comparison_df

In [ ]:
# رسم نمودار مقایسه زمان اجرا
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax.bar(comparison_df['روش'], comparison_df['زمان (ثانیه)'], 
              color=colors[:len(comparison_df)], alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('روش پردازش', fontsize=12)
ax.set_ylabel('زمان اجرا (ثانیه)', fontsize=12)
ax.set_title('مقایسه زمان اجرا در روش‌های مختلف پردازش', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, comparison_df['زمان (ثانیه)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.2f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

# محاسبه و نمایش speedup
if len(comparison_df) > 1:
    serial_time = comparison_df.iloc[0]['زمان (ثانیه)']
    for i in range(1, len(comparison_df)):
        speedup = serial_time / comparison_df.iloc[i]['زمان (ثانیه)']
        ax.text(i, comparison_df.iloc[i]['زمان (ثانیه)'] + 1, 
                f'Speedup: {speedup:.2f}x', ha='center', va='bottom', 
                fontsize=10, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۶. تأثیر تعداد هسته‌ها بر زمان اجرا
# ============================================================================

def test_different_workers(data_list, dist_name='Normal'):
    """تست تأثیر تعداد هسته‌ها بر زمان اجرا"""
    n_workers_list = [1, 2, 4, 8]
    times = []
    
    for n in n_workers_list:
        try:
            if n > cpu_count():
                continue
            start_time = time.time()
            _ = parallel_processing_multiprocessing(data_list, dist_name, n_workers=n)
            elapsed = time.time() - start_time
            times.append((n, elapsed))
            print(f"   {n} هسته: {elapsed:.2f} ثانیه")
        except Exception as e:
            print(f"   {n} هسته: خطا - {str(e)}")
    
    return times

print("\n📊 تأثیر تعداد هسته‌ها بر زمان اجرا:")
print("=" * 50)
worker_times = test_different_workers(test_data[:50], 'Normal')

if worker_times:
    fig, ax = plt.subplots(figsize=(10, 6))
    workers = [w for w, _ in worker_times]
    times = [t for _, t in worker_times]
    
    ax.plot(workers, times, 'bo-', linewidth=2, markersize=10, 
            markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('تعداد هسته‌ها', fontsize=12)
    ax.set_ylabel('زمان اجرا (ثانیه)', fontsize=12)
    ax.set_title('تأثیر تعداد هسته‌ها بر زمان اجرا', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    for x, y in zip(workers, times):
        ax.text(x, y + 0.1, f'{y:.2f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ مفهوم پردازش موازی و مزایای آن
✅ روش‌های مختلف پردازش موازی در Python
✅ استفاده از `multiprocessing` برای پردازش موازی
✅ استفاده از `joblib` برای پردازش موازی
✅ استفاده از `Dask` برای پردازش موازی
✅ مقایسه زمان اجرا در روش‌های مختلف
✅ تأثیر تعداد هسته‌ها بر زمان اجرا

---

**نکات کلیدی:**

1. پردازش موازی می‌تواند زمان اجرا را تا چندین برابر کاهش دهد.
2. انتخاب روش مناسب به نوع داده و پیچیدگی محاسبات بستگی دارد.
3. `multiprocessing` برای کنترل دقیق فرآیندها مناسب است.
4. `joblib` ساده‌ترین روش برای استفاده است.
5. `Dask` برای داده‌های بسیار بزرگ و سیستم‌های توزیع‌شده مناسب است.
6. افزایش تعداد هسته‌ها همیشه باعث افزایش سرعت نمی‌شود (سربار ارتباطی).

---

**مراحل بعدی:**
- نوت‌بوک ۰۸: مصورسازی داده
- نوت‌بوک ۰۹: افزودن توزیع سفارشی
- نوت‌بوک ۱۰: کاربرد پیشرفته